# Bias and Fairness in Machine Learning
## Module 5, Lesson 2

In this notebook, we will:
1. Generate a synthetic loan dataset with demographic attributes
2. Train a classifier and analyze bias
3. Compute fairness metrics (demographic parity, equal opportunity, equalized odds)
4. Apply a pre-processing bias mitigation technique

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

np.random.seed(42)

## 1. Generate Synthetic Loan Dataset

We create a dataset representing loan applications. The protected attribute is `race` (white vs. non-white). We simulate income and credit score differences to reflect real-world disparities.

In [ ]:
n = 5000

race = np.random.choice(['white', 'non_white'], size=n, p=[0.7, 0.3])

income = np.where(race == 'white',
                  np.random.normal(75, 25, n),
                  np.random.normal(50, 18, n))
income = np.clip(income, 15, 200)

credit_score = np.where(race == 'white',
                        np.random.normal(710, 50, n),
                        np.random.normal(650, 60, n))
credit_score = np.clip(credit_score, 300, 850)

loan_amount = np.random.uniform(1000, 50000, n)

prob_repayment = 0.5 + 0.003 * (credit_score - 600) / 250 + 0.002 * (income - 50) / 50
prob_repayment = np.clip(prob_repayment, 0.1, 0.99)
repayment = np.random.binomial(1, prob_repayment)

df = pd.DataFrame({
    'race': race,
    'income': income.round(1),
    'credit_score': credit_score.round(0),
    'loan_amount': loan_amount.round(0),
    'repayment': repayment
})

print("Dataset shape:", df.shape)
print("\nRace distribution:")
print(df['race'].value_counts(normalize=True))
print("\nRepayment rate by race:")
print(df.groupby('race')['repayment'].mean())

## 2. Train a Classifier

We train a logistic regression model to predict loan repayment.

In [ ]:
X = pd.get_dummies(df[['income', 'credit_score', 'loan_amount']], drop_first=True)
y = df['repayment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", (y_pred == y_test).mean())
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 3. Bias Analysis by Group

We compute fairness metrics by race.

In [ ]:
test_df = df.iloc[X_test.index].copy()
test_df['predicted'] = y_pred
test_df['probability'] = y_prob

def compute_fairness_metrics(subset, group_name, y_true_col='repayment', y_pred_col='predicted'):
    tn, fp, fn, tp = confusion_matrix(subset[y_true_col], subset[y_pred_col]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    approval_rate = subset[y_pred_col].mean()
    base_rate = subset[y_true_col].mean()
    return {
        'Group': group_name,
        'Size': len(subset),
        'Base Rate': base_rate,
        'Approval Rate': approval_rate,
        'TPR (Sensitivity)': tpr,
        'FPR': fpr,
        'TNR (Specificity)': tnr,
        'FNR': fnr
    }

results = []
for group in ['white', 'non_white']:
    subset = test_df[test_df['race'] == group]
    results.append(compute_fairness_metrics(subset, group))

fairness_df = pd.DataFrame(results).set_index('Group')
print("Fairness Metrics by Group:")
print(fairness_df.round(4))

In [ ]:
white_subset = test_df[test_df['race'] == 'white']
non_white_subset = test_df[test_df['race'] == 'non_white']

dp_diff = white_subset['predicted'].mean() - non_white_subset['predicted'].mean()
print(f"Demographic Parity Difference: {dp_diff:.4f}")
print(f"  (positive means white group has higher approval rate)")

white_tpr = white_subset['predicted'][white_subset['repayment'] == 1].mean()
non_white_tpr = non_white_subset['predicted'][non_white_subset['repayment'] == 1].mean()
print(f"\nEqual Opportunity (TPR) Difference: {white_tpr - non_white_tpr:.4f}")

white_fpr = white_subset['predicted'][white_subset['repayment'] == 0].mean()
non_white_fpr = non_white_subset['predicted'][non_white_subset['repayment'] == 0].mean()
print(f"False Positive Rate Difference: {white_fpr - non_white_fpr:.4f}")

disparate_impact = non_white_subset['predicted'].mean() / white_subset['predicted'].mean()
print(f"\nDisparate Impact Ratio: {disparate_impact:.4f}")
print(f"  (below 0.8 indicates adverse impact, EEOC standard)")

## 4. Visualizing Bias

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

metrics = ['Approval Rate', 'TPR (Sensitivity)', 'FPR']
colors = ['#4C72B0', '#DD8452']

for i, metric in enumerate(metrics):
    values = [fairness_df.loc['white', metric], fairness_df.loc['non_white', metric]]
    axes[i].bar(['White', 'Non-White'], values, color=colors)
    axes[i].set_title(metric)
    axes[i].set_ylim(0, 1)
    for j, v in enumerate(values):
        axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Bias Mitigation: Sample Reweighing

We implement a pre-processing technique: assign higher sample weights to underrepresented groups to balance the training data.

In [ ]:
train_df = df.iloc[X_train.index].copy()

white_mask = train_df['race'] == 'white'
non_white_mask = train_df['race'] == 'non_white'

w_white = len(train_df) / (2 * white_mask.sum())
w_non_white = len(train_df) / (2 * non_white_mask.sum())

sample_weights = np.where(white_mask, w_white, w_non_white)

print(f"Weight for white samples: {w_white:.4f}")
print(f"Weight for non-white samples: {w_non_white:.4f}")
print(f"Effective white weight sum: {sample_weights[white_mask].sum():.0f}")
print(f"Effective non-white weight sum: {sample_weights[non_white_mask].sum():.0f}")

In [ ]:
model_weighted = LogisticRegression(max_iter=1000, random_state=42)
model_weighted.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_weighted = model_weighted.predict(X_test)

test_df['predicted_weighted'] = y_pred_weighted

print("\nBefore Mitigation:")
before_white = test_df[test_df['race'] == 'white']['predicted'].mean()
before_non_white = test_df[test_df['race'] == 'non_white']['predicted'].mean()
print(f"  White approval rate: {before_white:.4f}")
print(f"  Non-white approval rate: {before_non_white:.4f}")
print(f"  Difference: {before_white - before_non_white:.4f}")

print("\nAfter Mitigation (Reweighing):")
after_white = test_df[test_df['race'] == 'white']['predicted_weighted'].mean()
after_non_white = test_df[test_df['race'] == 'non_white']['predicted_weighted'].mean()
print(f"  White approval rate: {after_white:.4f}")
print(f"  Non-white approval rate: {after_non_white:.4f}")
print(f"  Difference: {after_white - after_non_white:.4f}")

print("\nNote: Reweighing reduces demographic parity difference but may affect model accuracy.")

## 6. Exercises

### Basic
1. Change the threshold from 0.5 to 0.3 and 0.7. How do fairness metrics change?
2. Add `race` as a feature. Does the model become more or less fair?

### Implementation
3. Train a Random Forest on the same data. Compare fairness metrics with logistic regression.
4. Implement a function that computes all four fairness metrics (demographic parity, equal opportunity, equalized odds, disparate impact) for any model output.

### Critical Thinking
5. The reweighing technique balances group representation. What are its limitations? Could it introduce new biases?
6. If you were building a real loan approval system, how would you decide which fairness definition to use and which threshold to set?